<a href="https://colab.research.google.com/github/adi-devv/HateScan/blob/main/hate_speech_distilBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Colab_Projects/hateD/dataset/train.csv")


Mounted at /content/drive


In [5]:
# Colab-ready multi-label DistilBERT training

# Install transformers & datasets if not already installed
!pip install -q transformers datasets scikit-learn

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments

# ----------------------------
# 1️⃣ Load your dataset
# ----------------------------
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Create a column to indicate if it's hate (any label is 1)
df['is_hate'] = df[label_cols].sum(axis=1) > 0

# Separate hate and non-hate samples
hate_df = df[df['is_hate']]
non_hate_df = df[~df['is_hate']]

# Downsample non-hate to match hate entries (or some ratio, e.g., 1:1)
non_hate_downsampled = non_hate_df.sample(n=len(hate_df)*1, random_state=42)

# Combine
df_balanced = pd.concat([hate_df, non_hate_downsampled]).sample(frac=1, random_state=42)  # shuffle

# Optional: check
print("Balanced dataset shape:", df_balanced.shape)
print(df_balanced['is_hate'].value_counts())

train_df, val_df = train_test_split(df_balanced, test_size=0.2, random_state=42)

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

# ----------------------------
# 3️⃣ Tokenization
# ----------------------------
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['comment_text'], padding=True, truncation=True, max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

# ----------------------------
# 4️⃣ Format Labels
# ----------------------------
def format_labels(batch):
    batch_labels = []
    for i in range(len(batch['toxic'])):
        batch_labels.append([batch[col][i] for col in label_cols])
    batch['labels'] = np.array(batch_labels, dtype=np.float32)
    return batch

train_ds = train_ds.map(format_labels, batched=True)
val_ds = val_ds.map(format_labels, batched=True)

train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# ----------------------------
# 5️⃣ Load DistilBERT Model
# ----------------------------
num_labels = len(label_cols)

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# ----------------------------
# 6️⃣ Training Arguments
# ----------------------------
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    logging_dir='./logs',
    logging_steps=50,
    fp16=True if device=="cuda" else False,
    save_strategy="epoch",
    do_eval=True,
    report_to=[]  # disables WandB logging
)

# ----------------------------
# 7️⃣ Trainer Setup
# ----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer
)

# ----------------------------
# 8️⃣ Train & Evaluate
# ----------------------------
trainer.train()
results = trainer.evaluate()
print(results)

# ----------------------------
# 9️⃣ Prediction Function
# ----------------------------
def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits)
    return dict(zip(label_cols, probs.cpu().numpy()[0]))

# Test
print(predict("People like you don't belong here."))


Balanced dataset shape: (32450, 9)
is_hate
True     16225
False    16225
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Map:   0%|          | 0/6490 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-556160150.py:102: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
50,0.374700
100,0.228000
150,0.192200
200,0.185500
250,0.183600
300,0.183100
350,0.188400
400,0.177200
450,0.175700
500,0.171700


{'eval_loss': 0.22036729753017426, 'eval_runtime': 5.7472, 'eval_samples_per_second': 1129.246, 'eval_steps_per_second': 35.322, 'epoch': 5.0}
{'toxic': np.float32(0.6099889), 'severe_toxic': np.float32(3.347469e-05), 'obscene': np.float32(3.8229115e-05), 'threat': np.float32(0.00021318757), 'insult': np.float32(0.0013617034), 'identity_hate': np.float32(0.0008360903)}


In [7]:
# 1️⃣ Load fine-tuned model & tokenizer
# ----------------------------
# from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
# import torch

# device = "cuda" if torch.cuda.is_available() else "cpu"

# # path to your saved checkpoint
# model_path = "./results/checkpoint-7302"
# model = DistilBertForSequenceClassification.from_pretrained(model_path)
# tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)

# model.to(device)
# model.eval()

# ----------------------------
# 2️⃣ Prediction function (combine all labels)
# ----------------------------
def predict_hate(texts, threshold=0.0):
    """
    texts: list of strings
    threshold: sigmoid probability threshold for considering a label active
    returns: list of dicts: {all_probs_per_label, hate_label}
    """
    inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()  # shape: (batch_size, num_labels)

    results = []
    for prob in probs:
        # 1 if any class probability > threshold
        hate_label = int((prob >= threshold).any())
        results.append({"probabilities_per_label": prob, "hate_label": hate_label})
    return results

# ----------------------------
# 3️⃣ Test on 4 sentences
# ----------------------------
test_sentences = [
    "People like you don't belong here.",
    "I love everyone equally!",
    "You are a disgrace to humanity.",
    "Yo wassup nigga!",
    "I had a great time at the park today.",
    "The weather is really nice, let’s go for a walk.",
    "She is studying hard for her exams this week.",
    "I love watching movies on weekends with my friends.",
    "He is a very helpful and kind person.",
    "You are such a dumb idiot, can’t you do anything right?",
    "Shut the hell up, nobody cares about your opinion.",
    "Stop being such a jerk all the time.",
    "That was a stupid move, you really messed it up.",
    "Get lost loser, nobody likes you.",
    "People like you don’t deserve to be part of this group.",
    "I hate that community, they are all worthless.",
    "You don’t belong here, go back where you came from.",
    "Those people are disgusting and should stay away from us.",
    "Everyone in that group is trash and deserves nothing."
]


predictions = predict_hate(test_sentences, threshold=0.5)

for text, pred in zip(test_sentences, predictions):
    # round each probability to 2 decimals
    rounded_probs = [f"{p:.2f}" for p in pred['probabilities_per_label']]

    print(f"Text: {text}")
    print(f"Probabilities per label: {rounded_probs}")
    print(f"Hate label (1=hate, 0=not hate): {pred['hate_label']}")
    print("-" * 60)

Text: People like you don't belong here.
Probabilities per label: ['0.61', '0.00', '0.00', '0.00', '0.00', '0.00']
Hate label (1=hate, 0=not hate): 1
------------------------------------------------------------
Text: I love everyone equally!
Probabilities per label: ['0.00', '0.00', '0.00', '0.00', '0.00', '0.00']
Hate label (1=hate, 0=not hate): 0
------------------------------------------------------------
Text: You are a disgrace to humanity.
Probabilities per label: ['1.00', '0.00', '0.00', '0.00', '0.49', '0.00']
Hate label (1=hate, 0=not hate): 1
------------------------------------------------------------
Text: Yo wassup nigga!
Probabilities per label: ['0.97', '0.03', '0.88', '0.00', '0.74', '0.98']
Hate label (1=hate, 0=not hate): 1
------------------------------------------------------------
Text: I had a great time at the park today.
Probabilities per label: ['0.00', '0.00', '0.00', '0.00', '0.00', '0.00']
Hate label (1=hate, 0=not hate): 0
----------------------------------